# OfflineMedia Portable Video AI

GitHub-triggered Colab GPU worker for `CAption11/offlinemedia`. This notebook pulls the active branch, consumes a queued request, starts ComfyUI, generates a video, validates the output, and displays it in Colab.

In [ ]:
import os, subprocess, sys, pathlib
REPO='https://github.com/CAption11/offlinemedia.git'
BRANCH='claude/scan-repo-chatgpt-review-6nljgh'
ROOT=pathlib.Path('/content/offlinemedia')
if not ROOT.exists():
    subprocess.run(['git','clone','-b',BRANCH,REPO,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'reset','--hard',f'origin/{BRANCH}'],check=True)
os.chdir(ROOT)
print('Repository:',ROOT)
print('Branch:',BRANCH)

In [ ]:
import shutil, subprocess, sys
print('Python:',sys.version)
if not shutil.which('nvidia-smi'):
    raise RuntimeError('No NVIDIA GPU runtime. In Colab choose Runtime > Change runtime type > GPU.')
subprocess.run(['nvidia-smi'],check=False)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt','huggingface_hub'],check=True)
print('Dependencies installed.')

In [ ]:
GITHUB_TOKEN=os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
try:
    from google.colab import userdata
    if not GITHUB_TOKEN:
        GITHUB_TOKEN=userdata.get('GITHUB_TOKEN')
except Exception:
    pass
print('GitHub token configured:',bool(GITHUB_TOKEN))
if not GITHUB_TOKEN:
    print('WARNING: without GITHUB_TOKEN the queued job cannot be claimed or marked complete safely.')

In [ ]:
from portable.trigger_listener import read_trigger_or_none, wait_for_trigger
trigger, trigger_sha=read_trigger_or_none(token=GITHUB_TOKEN)
if trigger is None:
    print('Queue is empty. Seed a request or run the GitHub Actions trigger.')
else:
    print('Queued trigger:',trigger)
print('Waiting for a queued request...')
trigger=wait_for_trigger(token=GITHUB_TOKEN,poll_seconds=15,claim=bool(GITHUB_TOKEN))
print('Received trigger:',trigger)

In [ ]:
from portable.comfyui_bootstrap import bootstrap
comfy_process=bootstrap()
print('ComfyUI is running.')

In [ ]:
import json, subprocess, sys
mode=trigger.get('mode','text_to_video')
prompt=trigger.get('prompt','A small red ball rolling across a wooden table, natural lighting')
width=int(trigger.get('width',320)); height=int(trigger.get('height',240))
frames=int(trigger.get('frames',17)); fps=int(trigger.get('fps',8))
cmd=[sys.executable,'scripts/test_generation.py','--type',mode,'--prompt',prompt,'--width',str(width),'--height',str(height),'--frames',str(frames),'--fps',str(fps),'--workflow-dir','workflows/official']
print('Running:', ' '.join(cmd))
result=subprocess.run(cmd,text=True,capture_output=True)
print(result.stdout)
if result.stderr: print('STDERR:',result.stderr)
if result.returncode!=0: raise RuntimeError(f'Generation failed with exit code {result.returncode}')

In [ ]:
import json, re, shutil, subprocess
from pathlib import Path
from IPython.display import Video, display
paths=[]
for line in result.stdout.splitlines():
    if line.startswith('Output: '): paths.append(Path(line.split('Output: ',1)[1].strip()))
if not paths: raise RuntimeError('Generation returned no output path.')
for path in paths:
    if not path.is_file() or path.stat().st_size==0: raise RuntimeError(f'Invalid output file: {path}')
    print(f'Output: {path} ({path.stat().st_size:,} bytes)')
    ffprobe=shutil.which('ffprobe')
    if ffprobe:
        probe=subprocess.run([ffprobe,'-v','error','-print_format','json','-show_streams','-show_format',str(path)],text=True,capture_output=True,check=True)
        data=json.loads(probe.stdout)
        video=next((s for s in data.get('streams',[]) if s.get('codec_type')=='video'),None)
        if video is None: raise RuntimeError(f'No video stream in {path}')
        print('Validation:',json.dumps({'codec':video.get('codec_name'),'width':video.get('width'),'height':video.get('height'),'frames':video.get('nb_frames'),'fps':video.get('r_frame_rate'),'duration':(data.get('format') or {}).get('duration')},sort_keys=True))
    print('Displaying:',path)
    display(Video(str(path),embed=True))
print('REAL VIDEO GENERATION AND OUTPUT VALIDATION PASSED.')